# Контролируемые эксперименты v2: длина истории и признаки

GPU в Colab, затем четыре ячейки сверху вниз. Загрузи свежий `twotower_experiments.zip`.
Используются прежние `music_prompt/data/features.npz` и `data/pilot/` на Drive.

Каждый эксперимент — **новое обучение both с нуля от исходной предобученной Qwen**, 4 эпохи,
LoRA rank=8, E=2, те же 50 000 диалогов и эффективный batch=128. Старые обученные веса не нужны.

В последней ячейке выбери EXPERIMENTS:
- `history32`: последние 32 трека вместо 64, все три канала;
- `no_tags`: история 64, audio+text, без тегов;
- `no_audio`: история 64, text+tags, без аудио.

В каждом варианте меняется только указанный фактор относительно исходного both/64.
Сначала 20 шагов выбранных вариантов входят в обучение и дают замер скорости.
Если 4 эпохи не помещаются в HOURS, длинное обучение не начинается; результат не подменяется двухэпоховым.
Запуски последовательные, одна GPU. Checkpoint каждые 100 шагов, лучшие — по NDCG@100 на 5000 val-ходах.
Итоговая оценка — все 8421 val-хода.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

ROOT = Path('/content/music_prompt')
ROOT.mkdir(parents=True, exist_ok=True)
print('Выбери twotower_experiments.zip с компьютера')
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall(ROOT)
assert (ROOT / 'twotower/ablations.py').is_file(), 'Нужен новый архив twotower_experiments.zip'


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'twotower/requirements.txt')], check=True)


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/music_prompt')
source = DRIVE_ROOT / 'data'
if not (source / 'features.npz').is_file():
    source = DRIVE_ROOT / 'data/model'  # поддерживается и прежняя раскладка
assert (source / 'features.npz').is_file(), f'Нет {source / "features.npz"}'
assert (source / 'pilot/manifest.json').is_file(), f'Нет {source / "pilot/manifest.json"}'
local = ROOT / 'data/model'
local.mkdir(parents=True, exist_ok=True)
shutil.copy2(source / 'features.npz', local / 'features.npz')
shutil.copytree(source / 'pilot', local / 'pilot', dirs_exist_ok=True)
print('Данные скопированы на локальный диск Colab:', local)


In [ ]:
import json

EXPERIMENTS = ['history32']  # Следующий запуск; затем отдельно можно выбрать no_tags и no_audio
RUN_DIR = DRIVE_ROOT / 'runs' / 'ablations_v2_e2'
HOURS = 22  # бюджет этого вызова, включая оценку

gpu = subprocess.check_output([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Включи GPU"; print(torch.cuda.get_device_name())'], text=True).strip()
BATCH = 32 if 'A100' in gpu else 8
saved_batches = {json.loads((RUN_DIR / f'{job}_plan.json').read_text())['batch_size']
                 for job in EXPERIMENTS if (RUN_DIR / f'{job}_plan.json').exists()}
assert len(saved_batches) <= 1, 'Эти эксперименты начинались с разными batch: продолжай их отдельно'
if saved_batches:
    BATCH = saved_batches.pop()
print('GPU:', gpu, '| batch:', BATCH, '| эксперименты:', EXPERIMENTS, flush=True)
command = [sys.executable, '-u', '-m', 'twotower.ablations', '--jobs', *EXPERIMENTS,
           '--out', str(RUN_DIR), '--hours', str(HOURS), '--batch-size', str(BATCH)]
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
if status:
    raise RuntimeError(f'Остановка с кодом {status}; причина выше')
result = RUN_DIR / ('results_' + '_'.join(EXPERIMENTS) + '.json')
if result.exists():
    print(result.read_text())
    reference = DRIVE_ROOT / 'runs/lora_v2_e2/results.json'
    if reference.exists():
        baseline = json.loads(reference.read_text())['both']['metrics']
        print('Исходный both/64:', baseline)
else:
    print('Запуски ещё не завершены. Причина выше; checkpoint сохранены на Drive.')


## Результаты и продолжение

`MyDrive/music_prompt/runs/ablations_v2_e2/`:
- отдельная папка для каждого варианта с `config.json`, `metrics.jsonl`, `last.pt`, `best.pt`, `evaluation.json`;
- `*_plan.json` — параметры;
- `results_*.json` — итог выбранных запусков.

После обрыва повтори ячейки с теми же EXPERIMENTS и RUN_DIR. Сохранённые шаги повторно не считаются;
до 99 несохранённых шагов могут повториться. Новый вызов даёт новый бюджет HOURS.
Не меняй batch при resume. При смене A100 на T4 прежний batch=32 может не поместиться.

Не выбирай все варианты автоматически: начни с нужной проверки по результатам локального sweep.
Старые результаты both/text/history остаются в исходной папке.
